<a href="https://colab.research.google.com/github/naman-0804/learning/blob/Langchain/Langserve_api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
pip install "langserve[all]" fastapi uvicorn langchain-groq nest_asyncio

<frozen posixpath>:82: RuntimeWarning: coroutine 'Server.serve' was never awaited


In [14]:
from fastapi import FastAPI
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langserve import add_routes
from google.colab import userdata
import uvicorn

# 1. Setup API Key and Model
GROQ_API = userdata.get('GROQ_API_KEY')
model = ChatGroq(model="llama-3.3-70b-versatile", groq_api_key=GROQ_API)
parser = StrOutputParser()

# 2. Define the Chain from your notebook
generic_template = "Translate the following into {language}:"
prompt = ChatPromptTemplate.from_messages([
    ("system", generic_template),
    ("user", "{text}")
])

chain = prompt | model | parser

# 3. Create FastAPI app
app = FastAPI(
    title="LangChain Translation Server",
    version="1.0",
    description="Serving a translation chain using LangServe",
)

# 4. Add routes
add_routes(
    app,
    chain,
    path="/translate",
)

print("LangServe application is configured with the Translation chain at /translate.")
# To run: uvicorn.run(app, host="0.0.0.0", port=8000)

LangServe application is configured with the Translation chain at /translate.


In [16]:
import uvicorn
import nest_asyncio
import asyncio

nest_asyncio.apply()

async def run_server():
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
    server = uvicorn.Server(config)
    await server.serve()

if __name__ == "__main__":
    print("Starting server on http://0.0.0.0:8000")
    loop = asyncio.get_event_loop()
    loop.create_task(run_server())
    print("Server is running in the background loop.")

Starting server on http://0.0.0.0:8000
Server is running in the background loop.


In [18]:
from google.colab import output
# Ensure the proxy URL ends with a slash
proxy_url = output.eval_js("google.colab.kernel.proxyPort(8000)")
if not proxy_url.endswith('/'):
    proxy_url += '/'

print(f"Access your LangServe Playground at: {proxy_url}translate/playground")
print(f"Access API Docs at: {proxy_url}docs")

Access your LangServe Playground at: https://8000-m-s-kkb-use1d2-12hxtu3msocpa-d.us-east1-2.prod.colab.dev/translate/playground
Access API Docs at: https://8000-m-s-kkb-use1d2-12hxtu3msocpa-d.us-east1-2.prod.colab.dev/docs


In [ ]:
from langserve import RemoteRunnable

# This code talks to your server directly from within the notebook
# You don't need the web interface for this to work
remote_chain = RemoteRunnable(f"{proxy_url}translate")

# Change these values to test different translations
input_data = {
    "language": "German",
    "text": "I am testing my new translation server!"
}

print(f"Sending request: {input_data}")
result = remote_chain.invoke(input_data)
print(f"\nTranslation Result: {result}")